In [1]:
!nvidia-smi

Fri Oct 16 02:17:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!nvcc --version


nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Wed_Aug_20_01:58:59_PM_PDT_2025
Cuda compilation tools, release 13.0, V13.0.88
Build cuda_13.0.r13.0/compiler.36424714_0


In [3]:
%%writefile hello.cu
#include <cstdio>

__global__ void hello() {
    printf("Hello from GPU thread %d\n", threadIdx.x);
}

int main(void) {
    hello<<<1, 4>>>();
    cudaDeviceSynchronize();
    return 0;
}


Writing hello.cu


In [4]:
!nvcc -arch=sm_75 hello.cu -o hello && ./hello

Hello from GPU thread 0
Hello from GPU thread 1
Hello from GPU thread 2
Hello from GPU thread 3


In [5]:
%%writefile ex2_add.cu
#include <cstdio>

__global__ void add(int *a, int *b, int *c) {
    *c = *a + *b;
}

int main(void) {
    int a, b, c;
    int *d_a, *d_b, *d_c;
    int size = sizeof(int);

    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    a = 2;
    b = 7;

    cudaMemcpy(d_a, &a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, &b, size, cudaMemcpyHostToDevice);

    add<<<1, 1>>>(d_a, d_b, d_c);

    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess)
        printf("Kernel launch failed: %s\n", cudaGetErrorString(err));

    cudaMemcpy(&c, d_c, size, cudaMemcpyDeviceToHost);
    printf("Result is %d\n", c);

    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return 0;
}

Writing ex2_add.cu


In [6]:
!nvcc -arch=sm_75 ex2_add.cu -o ex2_add && ./ex2_add

Result is 9


In [7]:
%%writefile ex3_blocks.cu
#include <cstdio>
#include <cstdlib>

#define N 512

__global__ void add(int *a, int *b, int *c) {
    c[blockIdx.x] = a[blockIdx.x] + b[blockIdx.x];
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

int main(void) {
    int *a, *b, *c;
    int *d_a, *d_b, *d_c;
    int size = N * sizeof(int);

    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    a = (int*)malloc(size); random_ints(a, N);
    b = (int*)malloc(size); random_ints(b, N);
    c = (int*)malloc(size);

    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    add<<<N, 1>>>(d_a, d_b, d_c);

    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess)
        printf("Kernel launch failed: %s\n", cudaGetErrorString(err));

    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    int errors = 0;
    for (int r = 0; r < N; r++) {
        if (c[r] != a[r] + b[r]) errors++;
    }
    printf("Verification: %d mismatches out of %d\n", errors, N);

    free(a); free(b); free(c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return 0;
}

Writing ex3_blocks.cu


In [8]:
!nvcc -arch=sm_75 ex3_blocks.cu -o ex3_blocks && ./ex3_blocks

Verification: 0 mismatches out of 512


In [9]:
%%writefile ex4_threads.cu
#include <cstdio>
#include <cstdlib>

#define N 512

__global__ void addT(int *a, int *b, int *c) {
    c[threadIdx.x] = a[threadIdx.x] + b[threadIdx.x];
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

int main(void) {
    int *a, *b, *c;
    int *d_a, *d_b, *d_c;
    int size = N * sizeof(int);

    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    a = (int*)malloc(size); random_ints(a, N);
    b = (int*)malloc(size); random_ints(b, N);
    c = (int*)malloc(size);

    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    addT<<<1, N>>>(d_a, d_b, d_c);

    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess)
        printf("Kernel launch failed: %s\n", cudaGetErrorString(err));

    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    int errors = 0;
    for (int r = 0; r < N; r++) {
        if (c[r] != a[r] + b[r]) errors++;
    }
    printf("Verification: %d mismatches out of %d\n", errors, N);

    free(a); free(b); free(c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return 0;
}

Writing ex4_threads.cu


In [10]:
!nvcc -arch=sm_75 ex4_threads.cu -o ex4_threads && ./ex4_threads

Verification: 0 mismatches out of 512


In [11]:
%%writefile ex5_vecmul.cu
#include <cstdio>
#include <cstdlib>

#define N 10000000
#define THREADS_PER_BLOCK 512

__global__ void vecMul(int *a, int *b, int *c, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) {
        c[i] = a[i] * b[i];
    }
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

int main(void) {
    int *a, *b, *c;
    int *d_a, *d_b, *d_c;
    size_t size = (size_t)N * sizeof(int);

    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    a = (int*)malloc(size); random_ints(a, N);
    b = (int*)malloc(size); random_ints(b, N);
    c = (int*)malloc(size);

    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    int blocks = (N + THREADS_PER_BLOCK - 1) / THREADS_PER_BLOCK;
    printf("Launching %d blocks x %d threads\n", blocks, THREADS_PER_BLOCK);

    vecMul<<<blocks, THREADS_PER_BLOCK>>>(d_a, d_b, d_c, N);

    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess)
        printf("Kernel launch failed: %s\n", cudaGetErrorString(err));

    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    // Print the last 1000 results
    for (int r = N - 1000; r < N; r++) {
        printf("%d) %d * %d = %d\n", r, a[r], b[r], c[r]);
    }

    // Verify on CPU
    int errors = 0;
    for (int r = 0; r < N; r++) {
        if (c[r] != a[r] * b[r]) errors++;
    }
    printf("Verification: %d mismatches out of %d\n", errors, N);

    free(a); free(b); free(c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return 0;
}

Writing ex5_vecmul.cu


In [12]:
!nvcc -arch=sm_75 ex5_vecmul.cu -o ex5_vecmul && ./ex5_vecmul

Launching 19532 blocks x 512 threads
9999000) 76 * 52 = 3952
9999001) 76 * 74 = 5624
9999002) 41 * 94 = 3854
9999003) 8 * 0 = 0
9999004) 1 * 41 = 41
9999005) 28 * 52 = 1456
9999006) 45 * 4 = 180
9999007) 63 * 35 = 2205
9999008) 89 * 97 = 8633
9999009) 10 * 71 = 710
9999010) 74 * 91 = 6734
9999011) 65 * 41 = 2665
9999012) 29 * 60 = 1740
9999013) 53 * 5 = 265
9999014) 35 * 88 = 3080
9999015) 18 * 12 = 216
9999016) 17 * 12 = 204
9999017) 8 * 65 = 520
9999018) 51 * 34 = 1734
9999019) 79 * 64 = 5056
9999020) 13 * 89 = 1157
9999021) 10 * 85 = 850
9999022) 7 * 2 = 14
9999023) 31 * 51 = 1581
9999024) 88 * 86 = 7568
9999025) 59 * 8 = 472
9999026) 94 * 50 = 4700
9999027) 58 * 94 = 5452
9999028) 5 * 53 = 265
9999029) 36 * 67 = 2412
9999030) 80 * 45 = 3600
9999031) 33 * 5 = 165
9999032) 64 * 41 = 2624
9999033) 21 * 91 = 1911
9999034) 93 * 6 = 558
9999035) 65 * 34 = 2210
9999036) 1 * 43 = 43
9999037) 90 * 62 = 5580
9999038) 80 * 21 = 1680
9999039) 90 * 92 = 8280
9999040) 1 * 86 = 86
9999041) 54 * 1

In [13]:
%%writefile ex6_matrix.cu
#include <cstdio>
#include <cstdlib>

#define ROWS 10000
#define COLS 10000

__global__ void mulElem(int *a, int *b, int *c, int rows, int cols) {
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    int row = blockIdx.y * blockDim.y + threadIdx.y;

    if (row < rows && col < cols) {
        int idx = row * cols + col;
        c[idx] = a[idx] * b[idx];
    }
}

void random_ints(int *x, size_t size) {
    for (size_t i = 0; i < size; i++)
        x[i] = rand() % 100;
}

int main(void) {
    int *a, *b, *c;
    int *d_a, *d_b, *d_c;
    size_t count = (size_t)ROWS * COLS;
    size_t size = count * sizeof(int);

    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    a = (int*)malloc(size); random_ints(a, count);
    b = (int*)malloc(size); random_ints(b, count);
    c = (int*)malloc(size);

    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    dim3 threadsPerBlock(32, 16);
    dim3 numBlocks((COLS + threadsPerBlock.x - 1) / threadsPerBlock.x,
                   (ROWS + threadsPerBlock.y - 1) / threadsPerBlock.y);

    printf("Grid: %d x %d blocks\n", numBlocks.x, numBlocks.y);

    mulElem<<<numBlocks, threadsPerBlock>>>(d_a, d_b, d_c, ROWS, COLS);

    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess)
        printf("Kernel launch failed: %s\n", cudaGetErrorString(err));

    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    // Print C[9999][9000] through C[9999][9999]
    int r = 9999;
    for (int col = 9000; col < COLS; col++) {
        int idx = r * COLS + col;
        printf("C[%d][%d] = %d * %d = %d\n", r, col, a[idx], b[idx], c[idx]);
    }

    // Verify on CPU
    int errors = 0;
    for (size_t idx = 0; idx < count; idx++) {
        if (c[idx] != a[idx] * b[idx]) errors++;
    }
    printf("Verification: %d mismatches out of %zu\n", errors, count);

    free(a); free(b); free(c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return 0;
}

Writing ex6_matrix.cu


In [14]:
!nvcc -arch=sm_75 ex6_matrix.cu -o ex6_matrix && ./ex6_matrix

Grid: 313 x 625 blocks
C[9999][9000] = 29 * 56 = 1624
C[9999][9001] = 58 * 69 = 4002
C[9999][9002] = 76 * 50 = 3800
C[9999][9003] = 68 * 21 = 1428
C[9999][9004] = 67 * 78 = 5226
C[9999][9005] = 98 * 11 = 1078
C[9999][9006] = 3 * 29 = 87
C[9999][9007] = 69 * 52 = 3588
C[9999][9008] = 93 * 64 = 5952
C[9999][9009] = 60 * 65 = 3900
C[9999][9010] = 49 * 77 = 3773
C[9999][9011] = 5 * 44 = 220
C[9999][9012] = 6 * 77 = 462
C[9999][9013] = 17 * 31 = 527
C[9999][9014] = 45 * 65 = 2925
C[9999][9015] = 17 * 50 = 850
C[9999][9016] = 59 * 2 = 118
C[9999][9017] = 68 * 4 = 272
C[9999][9018] = 70 * 28 = 1960
C[9999][9019] = 54 * 11 = 594
C[9999][9020] = 7 * 87 = 609
C[9999][9021] = 29 * 93 = 2697
C[9999][9022] = 12 * 88 = 1056
C[9999][9023] = 14 * 26 = 364
C[9999][9024] = 74 * 6 = 444
C[9999][9025] = 24 * 52 = 1248
C[9999][9026] = 4 * 14 = 56
C[9999][9027] = 75 * 76 = 5700
C[9999][9028] = 0 * 40 = 0
C[9999][9029] = 65 * 7 = 455
C[9999][9030] = 67 * 13 = 871
C[9999][9031] = 29 * 96 = 2784
C[9999][9032] 